In [86]:
import sys
import os

# This makes imports work from the project root
project_root = os.path.abspath('..')   # go up one level from notebook
sys.path.insert(0, project_root)

from include.helper_functions import *

In [87]:
#read all csv
import os
import pandas as pd

# Define the path to the data directory
data_dir = "../data/raw/csv/ttc_subway_delay"

# List all CSV files in the data directory
csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

years_list = ['2023', '2024', '2025']
# Read each CSV file into a DataFrame based on year in filename and store in a list
dfs = []
for file in csv_files:
    # Extract year from filename (assuming format like "fafd_afdaf_2020.csv")   
    year = file.split('.')[0][-4:]  # Get the last 4 characters (the year)
    # Only process files for years in the specified list
    if str(year) in years_list:
        df = pd.read_csv(os.path.join(data_dir, file))
        df['year'] = year  # Add a column for the year
        dfs.append(df)
        print(f"Loaded {file}")

# Concatenate all DataFrames into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)
print("All CSV files have been combined into a single DataFrame.")

#drop id column
if 'id' in combined_df.columns:
    combined_df = combined_df.drop(columns=['id'])
    print("Dropped 'id' column from the combined DataFrame.")

#delete duplicate rows and print count of duplicate rows
duplicate_rows = combined_df[combined_df.duplicated(keep=False)]
print("Count of duplicate rows:", len(duplicate_rows))
combined_df = combined_df.drop_duplicates()
print("Removed duplicate rows from the combined DataFrame.")

#rename all the columns in combined_df as per dictionary
column_mapping = {
    'Date': 'date',
    'Time': 'time',
    'Day': 'day',
    'Station': 'station',
    'Code': 'code',
    'Min Delay': 'delayed_minutes',
    'Min Gap': 'gap_minutes',
    'Bound': 'bound',
    'Line': 'line',
    'Vehicle': 'vehicle'
}

# Rename columns in combined_df
combined_df = combined_df.rename(columns=column_mapping)

# Convert 'date' column to datetime
combined_df['date'] = pd.to_datetime(combined_df['date'])

# Display the first few rows of the combined DataFrame
print(combined_df.head())

# Extract hour of day (0–23) — captures patterns like rush hour
combined_df['hour'] = combined_df['date'].dt.hour

# Extract day of week as a number (0=Monday, 6=Sunday)
# Useful because weekdays typically have different delay patterns than weekends
combined_df['weekday'] = combined_df['date'].dt.weekday

# Binary flag: 1 if Saturday or Sunday, 0 otherwise
# Cast to int so it's stored as 0/1 instead of True/False
combined_df['is_weekend'] = (combined_df['weekday'] >= 5).astype(int)
# Extract month (1–12) — captures seasonal patterns (e.g. winter delays)
combined_df['month'] = combined_df['date'].dt.month

# Extract ISO week number (1–52) — finer-grained seasonal signal than month
combined_df['week'] = combined_df['date'].dt.isocalendar().week.astype(int)

#write in data/processed/combined_data.csv# Define the path to the output file
output_file = "../data/processed/ttc_subway_delay_data_combined.csv"
# Save the combined DataFrame to a new CSV file
combined_df.to_csv(output_file, index=False)
print(f"Combined data has been saved to {output_file}.")

Loaded ttc-subway-delay-data-2024.csv
Loaded ttc-subway-delay-data-2025.csv
Loaded ttc-subway-delay-data-2023.csv
All CSV files have been combined into a single DataFrame.
Dropped 'id' column from the combined DataFrame.
Count of duplicate rows: 160
Removed duplicate rows from the combined DataFrame.
        date   time     day             station   code  delayed_minutes  \
0 2024-01-01  02:00  Monday    SHEPPARD STATION    MUI                0   
1 2024-01-01  02:00  Monday      DUNDAS STATION   MUIS                0   
2 2024-01-01  02:08  Monday      DUNDAS STATION  MUPAA                4   
3 2024-01-01  02:13  Monday  KENNEDY BD STATION  PUTDN               10   
4 2024-01-01  02:22  Monday       BLOOR STATION  MUPAA                4   

   gap_minutes bound line  vehicle  year  
0            0     N   YU     5491  2024  
1            0     N   YU        0  2024  
2           10     N   YU     6051  2024  
3           16     E   BD     5284  2024  
4           10     N   YU     59

In [88]:
#read ttc station mapping file in processed folder
mapping_ttc_station_file = "../data/processed/mapping_ttc_station.csv"
mapping_ttc_station_df = pd.read_csv(mapping_ttc_station_file)

#read ttc delay code mapping file in processed folder
mapping_ttc_delay_code_file = "../data/processed/mapping_ttc_delay_code.csv"
mapping_ttc_delay_code_df = pd.read_csv(mapping_ttc_delay_code_file)

#read ttc subway station master file code mapping file in processed folder
mapping_ttc_subway_station_master_file = "../data/master/ttc_subway_station_master.csv"
mapping_ttc_subway_station_master_df = pd.read_csv(mapping_ttc_subway_station_master_file)


#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_station_df[mapping_ttc_station_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_station_df[mapping_ttc_station_df.duplicated(subset=['station'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)


#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_delay_code_df[mapping_ttc_delay_code_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_delay_code_df[mapping_ttc_delay_code_df.duplicated(subset=['delay_code'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)

#print count of duplicate rows
duplicate_rows_mapping = mapping_ttc_subway_station_master_df[mapping_ttc_subway_station_master_df.duplicated(keep=False)]
print("Count of duplicate rows in the mapping DataFrame:", len(duplicate_rows_mapping))

#find duplicate station names in the mapping file
duplicate_station_names = mapping_ttc_subway_station_master_df[mapping_ttc_subway_station_master_df.duplicated(subset=['station'], keep=False)]
print("Duplicate station names in the mapping DataFrame:")
print(duplicate_station_names)



Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [station, station_clean, row_count, mapped_station, score, include_station, remarks]
Index: []
Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [delay_code, row_count, mapped_delay_code, fuzzy_score, include_code, remarks]
Index: []
Count of duplicate rows in the mapping DataFrame: 0
Duplicate station names in the mapping DataFrame:
Empty DataFrame
Columns: [station, line, location, grade]
Index: []


In [90]:
#clean data for line and bound columns
combined_df["bound_clean"] = combined_df["bound"].apply(canonical_bound)

In [91]:
# Merge combined_df with mapping__ttc_station_df on 'station' column and rename columns
merged_df = pd.merge(combined_df, mapping_ttc_station_df, on='station', how='left') \
    .rename(columns={'score': 'station_score', 'remarks': 'station_remarks'}) \
    .drop(columns=['row_count', 'station_clean'])

# Merge merged_df with mapping__ttc_delay_code_df on 'code' column
merged_df = pd.merge(merged_df, mapping_ttc_delay_code_df, left_on='code', right_on='delay_code', how='left') \
    .rename(columns={'score': 'delay_code_score', 'remarks': 'delay_code_remarks', 'description': 'delay_code_description'}) \
    .drop(columns=['row_count'])

#rename the column from right side before merging with subway station master file
mapping_ttc_subway_station_master_df = mapping_ttc_subway_station_master_df.rename(columns={'station': 'subway_station', 'line': 'line_clean'})
merged_df = pd.merge(merged_df, mapping_ttc_subway_station_master_df, left_on='mapped_station', right_on='subway_station', how='left') \
    .drop(columns=['subway_station','location','grade'])
print("Merged DataFrame has been created by combining the combined DataFrame with the mapping DataFrames.")
#print the columns of merged_df
print("Columns in the merged DataFrame:")
print(merged_df.columns.tolist())


Merged DataFrame has been created by combining the combined DataFrame with the mapping DataFrames.
Columns in the merged DataFrame:
['date', 'time', 'day', 'station', 'code', 'delayed_minutes', 'gap_minutes', 'bound', 'line', 'vehicle', 'year', 'hour', 'weekday', 'is_weekend', 'month', 'week', 'bound_clean', 'mapped_station', 'station_score', 'include_station', 'station_remarks', 'delay_code', 'mapped_delay_code', 'fuzzy_score', 'include_code', 'delay_code_remarks', 'line_clean']


In [92]:
print(merged_df.shape)

(75048, 27)


In [83]:
# Identify outliers in 'Min Delay' and 'Min Gap' using MAD method within each line
# remove the rows with negative delay and 0 delay
merged_df = merged_df[merged_df["delayed_minutes"] > 0]

Z_THRESH = 3.5  # Adjust this threshold as needed
    

out_delay = merged_df["delayed_minutes"].transform(lambda s: mad_outlier_mask(s, z_thresh=Z_THRESH))

mask_outlier = out_delay 
print("\nRows flagged as outliers:", int(mask_outlier.sum()))

merged_df_clean = merged_df.loc[~mask_outlier].copy()
print("Final cleaned dataset shape:", merged_df_clean.shape)


Rows flagged as outliers: 2329
Final cleaned dataset shape: (24389, 27)


In [ ]:

#write merged_df to obt csv file in processed folder
output_file_merged = "../data/processed/ttc_subway_delay_data_obt.csv"
merged_df_clean.to_csv(output_file_merged, index=False)
print(f"Merged data has been saved to {output_file_merged}.")

Merged data has been saved to ../data/processed/ttc_subway_delay_data_obt.csv.


In [6]:
# read data from ttc_subway_delay_data_combined.csv into data frame and display the first few rows
import pandas as pd

# Define the path to the input file
input_file = "../data/processed/ttc_subway_delay_data_obt.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(input_file)

# Display the first few rows of the DataFrame
print(df.head())


         date   time     day             station   code  delayed_minutes  \
0  2024-01-01  02:08  Monday      DUNDAS STATION  MUPAA                4   
1  2024-01-01  02:13  Monday  KENNEDY BD STATION  PUTDN               10   
2  2024-01-01  02:22  Monday       BLOOR STATION  MUPAA                4   
3  2024-01-01  02:25  Monday    ST CLAIR STATION  MUPAA                3   
4  2024-01-01  02:27  Monday    WOODBINE STATION   EUDO                7   

   gap_minutes bound line  vehicle  ...  bound_clean  mapped_station  \
0           10     N   YU     6051  ...            N     Dundas West   
1           16     E   BD     5284  ...            E         Kennedy   
2           10     N   YU     5986  ...            N     Bloor–Yonge   
3            9     N   YU     6051  ...            N       St. Clair   
4           13     E   BD     5077  ...            E        Woodbine   

   station_score  include_station           station_remarks  delay_code  \
0             90              1.0  

In [8]:
#print distince lines in the data frame
distinct_lines = df['line_clean'].value_counts()
print(distinct_lines)

line_clean
YU         14117
BD          9185
SHP          807
SRT          279
UNKNOWN        1
Name: count, dtype: int64


In [52]:
#get distinct codes with count from the data frame
distinct_codes = df['mapped_delay_code'].value_counts()
print(distinct_codes)

mapped_delay_code
SUDP     3396
MUPAA    2343
PUOPO    2100
MUIR     1513
SUO      1153
         ... 
PUDCS       1
PUTCD       1
MUCP        1
TUST        1
PUMST       1
Name: count, Length: 123, dtype: int64
